# Fase de Modelação – adaptação das demos do professor aos dados da UCIN (Grupo 5)

Este notebook segue a estrutura das três demos do professor (`demo_class`, `demo_reg`, `demo_clustering`), mas usa os conjuntos de dados que o grupo preparou em `data/processed/`:

| Objetivo | Demo | Ficheiro | Unidade |
|---|---|---|---|
| O1 – Classificação de patologias | `demo_class` | `dataset_classificacao.csv` | bebé × dia (449) |
| O2 – Tempo de internamento | `demo_reg` | `dataset_regressao.csv` | bebé, primeiras 24 h (47) |
| O3 – Perfis de bebés | `demo_clustering` | `dataset_clustering.csv` | bebé, internamento completo (47) |

**Duas diferenças face às demos**, justificadas pelos dados:
1. **Validação por bebé.** No O1 há vários dias do mesmo bebé; se o `train_test_split` misturar dias do mesmo bebé no treino e no teste, os resultados ficam inflacionados. Usamos `GroupShuffleSplit` / `StratifiedGroupKFold` com o episódio como grupo.
2. **Validação cruzada repetida.** Com 47 bebés, uma só partição treino/teste depende muito da sorte; repetimos a validação e comparamos sempre com um *baseline* (Dummy).

## 0. Instalar e importar módulos

In [ ]:
# !pip install pandas scikit-learn matplotlib seaborn graphviz

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import tree
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import (GroupShuffleSplit, StratifiedGroupKFold,
                                     KFold, RepeatedKFold, train_test_split)
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_auc_score, f1_score, recall_score, cohen_kappa_score,
                             mean_absolute_error, root_mean_squared_error, silhouette_score)

SEED = 12345
import os
DATA = next(p for p in ["data/processed", "../data/processed"] if os.path.exists(p))   # pasta onde os notebooks 01/02 gravam os CSV

---
# O1 – Classificação de patologias (`demo_class`)
## 1.1 Carregar o conjunto de dados

In [ ]:
cl = pd.read_csv(f"{DATA}/dataset_classificacao.csv")
print(cl.shape, "| bebés:", cl.episodio.nunique())
cl.head(3)

O alvo é multi-rótulo (um dia pode ter várias patologias). Tratamos **um classificador por patologia**. Vamos focar-nos em **DMH** e **icterícia**, que são as do objetivo O1; anemia, CMV e "outra" têm poucos casos.

In [ ]:
alvos = ["pat_dmh", "pat_ictericia", "pat_anemia", "pat_cmv", "pat_outra"]
print((cl[alvos].mean() * 100).round(1).rename("% de dias com a patologia"))
print("\nDias com valores em falta (1.º dia de cada bebé):", int(cl.sat_o2_delta.isna().sum()))

## 1.2 Variáveis explicativas e separação treino/teste (por bebé)
Excluímos o identificador, a data e os alvos. Os valores em falta (variações no 1.º dia do bebé) são imputados **dentro do pipeline**, só com dados de treino.

In [ ]:
X_cols = [c for c in cl.columns if c not in ["episodio", "data"] + alvos]
X = cl[X_cols]
grupos = cl["episodio"]
print(len(X_cols), "variáveis explicativas")

# Partição 70/30 como na demo, mas sem separar dias do mesmo bebé
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
i_tr, i_ts = next(gss.split(X, groups=grupos))
print("treino:", len(i_tr), "dias /", grupos.iloc[i_tr].nunique(), "bebés | teste:", len(i_ts), "dias /", grupos.iloc[i_ts].nunique(), "bebés")

## 1.3 Demo: árvore de decisão, rede neuronal e random forest (uma só partição)
Igual à demo do professor, aplicada à **icterícia** e ao **DMH**.

In [ ]:
def modelos_class():
    return {
        "Baseline (prevalência)": DummyClassifier(strategy="prior"),
        "Árvore (max_depth=3)": make_pipeline(SimpleImputer(strategy="median"),
                                          DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=SEED)),
        "MLP [8,4]": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                   MLPClassifier(hidden_layer_sizes=[8, 4], max_iter=2000, alpha=1e-2, random_state=SEED)),
        "Random Forest": make_pipeline(SimpleImputer(strategy="median"),
                                       RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced", random_state=SEED)),
    }

for alvo in ["pat_ictericia", "pat_dmh"]:
    y = cl[alvo]
    print(f"\n===== {alvo} =====")
    for nome, m in modelos_class().items():
        m.fit(X.iloc[i_tr], y.iloc[i_tr])
        p = m.predict(X.iloc[i_ts])
        print(f"{nome:24s} accuracy={accuracy_score(y.iloc[i_ts], p):.3f}  F1-macro={f1_score(y.iloc[i_ts], p, average='macro'):.3f}")

In [ ]:
# Relatório detalhado de um modelo (como na demo): Random Forest para a icterícia
y = cl["pat_ictericia"]
rf = modelos_class()["Random Forest"].fit(X.iloc[i_tr], y.iloc[i_tr])
print(classification_report(y.iloc[i_ts], rf.predict(X.iloc[i_ts]), target_names=["sem icterícia", "icterícia"]))

**Atenção:** estes números vêm de **uma única partição** com poucos bebés no teste, por isso variam muito com a semente. A secção seguinte repete a validação várias vezes.

## 1.4 Avaliação robusta: validação cruzada repetida, agrupada por bebé
`StratifiedGroupKFold` (5 partições × 5 repetições). As métricas são calculadas sobre todas as previsões fora-da-amostra de cada repetição e depois fazemos a média das repetições.

In [ ]:
def cv_classificacao(fabrica, y, reps=5, k=5):
    linhas = []
    for r in range(reps):
        proba = np.zeros(len(y))
        pred = np.zeros(len(y), dtype=int)
        cv = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=SEED + r)
        for tr, ts in cv.split(X, y, grupos):
            m = fabrica().fit(X.iloc[tr], y.iloc[tr])
            pred[ts] = m.predict(X.iloc[ts])
            proba[ts] = m.predict_proba(X.iloc[ts])[:, 1]
        linhas.append(dict(
            AUC=roc_auc_score(y, proba),
            F1_macro=f1_score(y, pred, average="macro", zero_division=0),
            kappa=cohen_kappa_score(y, pred),
            recall=recall_score(y, pred, zero_division=0)))
    return pd.DataFrame(linhas).mean()

resultados_o1 = {}
for alvo in ["pat_ictericia", "pat_dmh"]:
    y = cl[alvo]
    tab = {nome: cv_classificacao(lambda n=nome: modelos_class()[n], y) for nome in modelos_class()}
    resultados_o1[alvo] = pd.DataFrame(tab).T.round(3)
    print(f"\n===== {alvo} (prevalência {y.mean():.1%}) =====")
    display(resultados_o1[alvo])

**Metas da Fase 1 (Tabela 8):** F1-macro ≥ 0,50 · ROC-AUC ≥ 0,70 · kappa ≥ 0,40 · recall ≥ 0,70.
Compara cada linha com estas metas e com o *baseline* (AUC 0,50 e kappa 0 por definição).

## 1.5 Interpretação: árvore de decisão e importância das variáveis

In [ ]:
y = cl["pat_ictericia"]
arv = make_pipeline(SimpleImputer(strategy="median"),
                    DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=SEED)).fit(X, y)
plt.figure(figsize=(18, 7))
tree.plot_tree(arv[-1], feature_names=X_cols, class_names=["não", "icterícia"], filled=True, rounded=True, fontsize=9)
plt.title("Árvore de decisão (profundidade 3) – icterícia")
plt.show()

In [ ]:
rf_full = make_pipeline(SimpleImputer(strategy="median"),
                        RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced", random_state=SEED)).fit(X, y)
imp = pd.Series(rf_full[-1].feature_importances_, index=X_cols).sort_values(ascending=False).head(10)
imp[::-1].plot.barh(figsize=(7, 4), title="Random Forest – 10 variáveis mais importantes (icterícia)")
plt.tight_layout(); plt.show()

---
# O2 – Tempo de internamento (`demo_reg`)
## 2.1 Carregar o conjunto de dados

In [ ]:
rg = pd.read_csv(f"{DATA}/dataset_regressao.csv")
print(rg.shape)
print(rg["los"].describe().round(1))
print("Internamentos cortados à direita:", int(rg.cens_direita.sum()), "de", len(rg))

## 2.2 Variáveis explicativas e alvo
Só usamos informação das **primeiras 24 h**. Excluímos `los`, `log_los` (alvo) e `cens_direita` / `cens_esquerda`, que só se conhecem no fim do internamento.

In [ ]:
Xr_cols = [c for c in rg.columns if c not in ["episodio", "los", "log_los", "cens_direita", "cens_esquerda"]]
Xr, yr = rg[Xr_cols], rg["los"]
print(len(Xr_cols), "variáveis explicativas")

## 2.3 Demo: partição 70/30, rede neuronal e métricas (como na demo)

In [ ]:
Xtr, Xts, ytr, yts = train_test_split(Xr, yr, test_size=0.30, random_state=SEED)
nn = make_pipeline(StandardScaler(),
                   MLPRegressor(hidden_layer_sizes=[8, 4], max_iter=5000, random_state=SEED, alpha=1e-1)).fit(Xtr, ytr)
pred = nn.predict(Xts)
print(f"MAE = {mean_absolute_error(yts, pred):.2f} dias")
print(f"NMAE = {mean_absolute_error(yts, pred) / (yts.max() - yts.min()) * 100:.1f} %")
print(f"RMSE = {root_mean_squared_error(yts, pred):.2f} dias")

## 2.4 Avaliação robusta: validação cruzada repetida (5 partições × 10 repetições)
Comparamos com um **baseline** que prevê sempre a mediana do treino.

In [ ]:
def modelos_reg():
    return {
        "Baseline (mediana)": DummyRegressor(strategy="median"),
        "Árvore (max_depth=2)": DecisionTreeRegressor(max_depth=2, min_samples_leaf=5, random_state=SEED),
        "MLP [8,4]": make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=[8, 4], max_iter=5000, alpha=1e-1, random_state=SEED)),
        "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=SEED),
    }

def cv_regressao(fabrica, reps=10, k=5):
    maes, rmses = [], []
    for r in range(reps):
        pred = np.zeros(len(yr))
        for tr, ts in KFold(k, shuffle=True, random_state=SEED + r).split(Xr):
            pred[ts] = fabrica().fit(Xr.iloc[tr], yr.iloc[tr]).predict(Xr.iloc[ts])
        maes.append(mean_absolute_error(yr, pred)); rmses.append(root_mean_squared_error(yr, pred))
    return dict(MAE=np.mean(maes), DP_MAE=np.std(maes), RMSE=np.mean(rmses))

tab = {n: cv_regressao(lambda n=n: modelos_reg()[n]) for n in modelos_reg()}
res_o2 = pd.DataFrame(tab).T
res_o2["NMAE_%"] = res_o2["MAE"] / (yr.max() - yr.min()) * 100
base = res_o2.loc["Baseline (mediana)", "MAE"]
res_o2["melhoria_vs_baseline_%"] = (1 - res_o2["MAE"] / base) * 100
res_o2.round(2)

**Meta da Fase 1:** MAE ≤ 6 dias. Repara na coluna `melhoria_vs_baseline_%`: é a que diz se o modelo aprende alguma coisa além de prever a mediana.

### Efeito da sorte na partição única
Repetindo a partição 70/30 da demo com 200 sementes diferentes, vê-se como o MAE de uma só partição é instável:

In [ ]:
maes = []
for s in range(200):
    a, b, c, d = train_test_split(Xr, yr, test_size=0.30, random_state=s)
    maes.append(mean_absolute_error(d, RandomForestRegressor(n_estimators=100, min_samples_leaf=2, random_state=0).fit(a, c).predict(b)))
print(f"MAE da Random Forest em 200 partições 70/30: média {np.mean(maes):.2f}, mínimo {np.min(maes):.2f}, máximo {np.max(maes):.2f}")
plt.hist(maes, bins=20); plt.xlabel("MAE (dias)"); plt.title("MAE da Random Forest em 200 partições 70/30"); plt.show()

---
# O3 – Perfis de bebés (`demo_clustering`)
## 3.1 Carregar o conjunto de dados

In [ ]:
ct = pd.read_csv(f"{DATA}/dataset_clustering.csv")
print(ct.shape)
# los_validacao e cens_direita ficam de fora: servem só para validar os clusters depois
Xc_cols = [c for c in ct.columns if c not in ["episodio", "los_validacao", "cens_direita"]]
Xc = ct[Xc_cols]
Xc_z = StandardScaler().fit_transform(Xc)        # K-Means usa distâncias: é preciso normalizar
print(len(Xc_cols), "variáveis")

## 3.2 Explorar (pairplot, como na demo)

In [ ]:
sns.pairplot(ct[["sat_o2_media", "puls_rate_media", "perfusion_media", "n_patologias_distintas"]], diag_kind="hist")
plt.show()

## 3.3 Escolher o número de clusters: método do cotovelo e silhueta

In [ ]:
ks = range(2, 9)
inercia, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(Xc_z)
    inercia.append(km.inertia_); sil.append(silhouette_score(Xc_z, km.labels_))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(list(ks), inercia, "o-"); ax[0].set_title("Cotovelo (inércia)"); ax[0].set_xlabel("k")
ax[1].plot(list(ks), sil, "o-"); ax[1].axhline(0.25, ls="--", c="gray"); ax[1].set_title("Silhueta (meta > 0,25)"); ax[1].set_xlabel("k")
plt.tight_layout(); plt.show()
pd.Series(sil, index=list(ks), name="silhueta").round(3)

## 3.4 K-Means final e perfis
Escolha o `K` com base no gráfico acima (cotovelo + silhueta + interpretabilidade). Começamos com 3.

In [ ]:
K = 3
km = KMeans(n_clusters=K, n_init=20, random_state=SEED).fit(Xc_z)
ct["cluster"] = km.labels_
print("Silhueta:", round(silhouette_score(Xc_z, km.labels_), 3))
print(ct["cluster"].value_counts().sort_index().rename("n.º de bebés"))
perfil_cols = ["sat_o2_media", "puls_rate_media", "perfusion_media", "pct_horas_sat_lt90", "pct_horas_perf_baixa",
               "frac_dias_pat_dmh", "frac_dias_pat_ictericia", "n_patologias_distintas", "los_validacao"]
ct.groupby("cluster")[perfil_cols].mean().round(2)

In [ ]:
# Visualização em 2D (PCA) apenas para ver a separação dos grupos
pc = PCA(2, random_state=SEED).fit_transform(Xc_z)
plt.scatter(pc[:, 0], pc[:, 1], c=ct["cluster"], cmap="viridis")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title(f"K-Means (k={K}) projetado em 2 componentes principais")
plt.show()

### Validação externa
Os clusters foram formados **sem** o tempo de internamento. Se tiverem sentido clínico, devem diferir também nessa variável:

In [ ]:
ct.boxplot(column="los_validacao", by="cluster"); plt.title(""); plt.suptitle("")
plt.ylabel("Dias de internamento"); plt.show()

---
# Resumo dos resultados face às metas da Fase 1
(valores obtidos com `SEED = 12345`; podem variar ligeiramente com outra semente)

| Objetivo | Meta | Resultado | Cumprida? |
|---|---|---|---|
| O1 – icterícia | F1-macro ≥ 0,50 · AUC ≥ 0,70 · kappa ≥ 0,40 · recall ≥ 0,70 | MLP: F1-macro 0,71 · AUC 0,78 · kappa 0,41 · recall 0,58 | Parcialmente (recall abaixo) |
| O1 – DMH | idem | melhor modelo (RF): AUC 0,62 · kappa 0,14 | Não: pouco acima do acaso |
| O2 – internamento | MAE ≤ 6 dias | baseline (mediana) 6,9 dias; nenhum modelo o bate | Não |
| O3 – clusters | silhueta > 0,25 | 0,12 a 0,15 | Não (mas os clusters diferem no tempo de internamento) |

**Leitura:** só a icterícia mostra sinal útil nos sinais vitais. O DMH e o tempo de internamento, com 47 bebés, não são previsíveis de forma fiável a partir destas variáveis. Isto é um resultado válido e deve ser reportado com honestidade, com o baseline ao lado.